# CANDLE V4 continuous_class_v1 — experimental variant

Same original TRAIN, current-task filtering, backbone loss/optimizer/steps, graph,
100 clients, seed42, six tasks x20 rounds and final T5/R19 TEST protocol.
V1 source and notebook remain unchanged. No validation/roles/shadow model.

The V4-only router keeps continuous selected activations and class/task identity
from the SAME native reference samples, capped by the existing per-class budget.
It does not retain raw inputs. Native binary memory/capsules and RNG are preserved.
Scaler and class-balanced LR fit only saved feature memory; class predictions map
to task IDs. No true task at inference. Expert handoff and scoped threshold .5 remain.

Native V1 multiclass remains a separate unchanged evaluation control on identical
TEST rows. Continuous-native and expert outputs are recorded without fitting TEST.
No extra per-round/per-task evaluation. Logging stays compact.

This variant has technical regression only, NO measured benchmark gain.
Fresh current-task memory capture is required: binary checkpoints cannot convert.
Keep RESUME_STATE empty for a new run; old v4/v5/v6/v7 runs cannot resume as v8.
Enable GPU + Internet. Use Kaggle Save Version to retain checkpoint/output.

## V4 bootstrap state fix (protocol v8)

At an exact representative model clone, copy the matching continuous memory,
scaler and classifier without sharing array references. Preserve inherited tasks;
do not copy routing memory during partial catch-up. Record provenance and bytes.
No change to V1, raw TRAIN, loss, native optimizer/steps, graph, fusion threshold
or final TEST protocol. Existing v7 notebooks/results remain unchanged.

Fresh run: RESUME_STATE="". Do not resume results16/17 as v8: their missing state
is already absent and this fix is not a retroactive checkpoint migration.
Technical regressions only; no measured Accuracy gain from the fix yet.


In [ ]:
# CANDLE V4 coverage-aware full development. Edit paths only.
import os
from pathlib import Path
import shutil
import subprocess
import sys
import zipfile

CODE_REF = "5221f52f4fa1609c70c7f344ea2370c50afb6d00"
DATA_DIR = "/kaggle/input/datasets/khoilv2005/100-clients"
OUTPUT_DIR = "/kaggle/working/results_candle_v4_continuous_bootstrap_seed42"
VARIANT = "coverage"  # V4 only. Keep the separate V1 notebook unchanged.
ROUTER_MODE = "continuous_class_v1"  # Locked in config; binary_cosine remains an explicit variant.
# Optional mounted round continuation .pt or task archive .zip; empty=fresh T0.
# Resume needs the same bootstrap-v8 run lock and router configuration.
RESUME_STATE = ""

if not Path('/kaggle/working').is_dir():
    raise RuntimeError('Run this notebook on Kaggle, not locally')
env = {**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"}
for name in ('DENICE_CONFIG_FILE','DENICE_CONFIG_OVERRIDES','DENICE_RESUME_STATE_PATH',
             'DENICE_CONFIG_ONLY','DENICE_TASK_END'):
    env.pop(name,None)
code = Path('/kaggle/working/candle_v4_bootstrap_source')
if not code.exists():
    subprocess.check_call(['git','clone','--filter=blob:none','--no-checkout',
        'https://github.com/khoilv2005/FL_IL_IDS.git',str(code)],env=env)
    subprocess.check_call(['git','sparse-checkout','set','--no-cone','/candle_v4_runtime/'],cwd=code,env=env)
    subprocess.check_call(['git','checkout','--detach',CODE_REF],cwd=code,env=env)
actual = subprocess.check_output(['git','rev-parse','HEAD'],cwd=code,text=True).strip()
if actual != CODE_REF:raise ValueError('Source pin mismatch; use a fresh source folder')
runtime = code/'candle_v4_runtime'
subprocess.check_call([sys.executable,'-m','pip','install','-q','-r',
                       str(runtime/'requirements-candle-v1-kaggle.txt')],env=env)
args = [sys.executable,'-u',str(runtime/'tools/train_candle_v4.py'),
        '--data-dir',DATA_DIR,'--output',OUTPUT_DIR,'--variant',VARIANT,'--router-mode',ROUTER_MODE]
args += ['--continuous-bootstrap']
if RESUME_STATE:
    source = Path(RESUME_STATE);out=Path(OUTPUT_DIR);out.mkdir(parents=True,exist_ok=True)
    if source.suffix=='.zip':
        with zipfile.ZipFile(source) as archive:
            for name in archive.namelist():
                relative=Path(name)
                if relative.is_absolute() or '..' in relative.parts:raise ValueError('Unsafe resume archive')
                if name=='candle_v4_run_lock.json' or name.startswith(('v4_threshold_locks/','v4_gate_journal/')):
                    target=out/relative;target.parent.mkdir(parents=True,exist_ok=True)
                    target.write_bytes(archive.read(name))
    else:
        if source.parent.resolve()!=out.resolve():
            shutil.copy2(source.parent/'candle_v4_run_lock.json',out/'candle_v4_run_lock.json')
        if source.parent.resolve()!=out.resolve() and (source.parent/'v4_threshold_locks').is_dir():
            shutil.copytree(source.parent/'v4_threshold_locks',out/'v4_threshold_locks',dirs_exist_ok=True)
        if source.parent.resolve()!=out.resolve() and (source.parent/'v4_gate_journal').is_dir():
            shutil.copytree(source.parent/'v4_gate_journal',out/'v4_gate_journal',dirs_exist_ok=True)
    args += ['--resume',RESUME_STATE]
print('Pinned source:',actual,flush=True)
print('Full TEST runs only after Task5/Round19 consolidation; retain output with Kaggle Save Version.',flush=True)
subprocess.check_call(args,cwd=runtime,env=env)
import json
completion=json.loads(Path(OUTPUT_DIR,'candle_v1_completion.json' if VARIANT=='native-only' else 'candle_v4_completion.json').read_text())
print('Completed:',completion['completed'],'metrics:',
      {arm:{k:m[k] for k in ('accuracy','macro_f1')} for arm,m in completion['final_test'].items()})
